# Data Orchestration with Apache Airflow

In this lesson, we will explore **Apache Airflow**, the industry standard for workflow orchestration, and configure it using Docker Compose.

## 1. Core Airflow Architecture Concepts

Apache Airflow represents pipelines as **Directed Acyclic Graphs (DAGs)**. When deploying Airflow in Docker, the architecture is broken down into separate specialized services:

1. **PostgreSQL (`postgres`)**: Airflow's internal metadata database. It stores pipeline run histories, task execution states, connection credentials, and user account configurations.

2. **Airflow API Server (`airflow-apiserver`)**: Manages external REST API calls and internal communication between worker processes and the metadata database.

3. **Airflow Scheduler (`airflow-scheduler`)**: The brain of the orchestrator. It monitors DAG files, tracks execution states, and triggers tasks as their time window schedules or dependencies are met.

4. **Airflow DAG Processor (`airflow-dag-processor`)**: Isolates the DAG parsing step. It continuously reads DAG files to check for changes and register them with the scheduler without slowing down execution.

5. **Airflow Triggerer (`airflow-triggerer`)**: Runs trigger loops to support deferrable (asynchronous) operators, allowing long-running tasks to yield resources while waiting.

6. **Airflow Initialization (`airflow-init`)**: A one-time setup container that migrates the metadata database, initializes the admin user credentials, and ensures directory file permissions (`AIRFLOW_UID`) are correctly configured.

## 2. Transitioning from Seeds to Ingestion

Up to this point, we have relied on a static file (`seeds/token_prices.csv`) loaded via `dbt seed` to calculate the USD value of our token transfers. However, token prices change by the second. Manually updating and redeploying a CSV file is not viable in a production data pipeline.

To automate this, we introduced the Python script `ingest_prices.py`. Instead of static seeds, this script dynamically queries historical hourly prices from the DefiLlama API and inserts them directly into the database.

Because we are replacing our static CSV seed with a live database feed, we must make three key adjustments to our dbt project:

### A. Remove the Static Seed
Delete the static CSV file `token_prices.csv` from the `seeds/` directory of your dbt project. We will no longer compile or load this seed file.

### B. Register the Live Table as a Source
Since the price ingestion script writes the data directly to the database, dbt needs to treat `token_prices` as an external database source rather than an internal dbt-managed seed.

Add the `token_prices` table under the source configuration inside `models/staging/sources.yml`:
```yaml
sources:
  - name: dataset
    schema: analytics
    tables:
      - name: Transfer
      - name: token_prices  # Registering the table populated by the Python script
```

### C. Update the Analytical Mart Models
Finally, update your dbt models to read from the live source database instead of the seed reference.

In `models/mart/fct_transfers.sql`, update the reference:
* **Old (using seeds)**:
  ```sql
  select * from {{ ref('token_prices') }}
  ```
* **New (using live source)**:
  ```sql
  select * from {{ source('dataset', 'token_prices') }}
  ```

## 3. Configuring Airflow Environment and Dependencies

To configure the containerized Airflow stack successfully, you must create two key configuration files in your `airflow/` root directory: `.env` and `requirements.txt`.

---

### A. The Environment File (`.env`)
The `.env` file contains the database connection credentials and system configuration variables.

Create a file named `.env` inside your `airflow/` directory and add the following content:
```bash
ENVIO_PG_DATABASE=data-set
ENVIO_PG_USER=root
ENVIO_PG_PASSWORD=root
ENVIO_PG_PORT=5432
ENVIO_PG_PUBLIC_SCHEMA=analytics
ENVIO_PG_HOST=db
AIRFLOW_UID=1000
```

---

### B. The Python Dependencies File (`requirements.txt`) | find code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-04/files/requirements.txt)
Because Airflow will execute our custom ingestion script and run `dbt build` commands, the container environment must have the necessary Python drivers and libraries installed.

Create a file named `requirements.txt` inside your `airflow/` directory and add these dependencies:
```bash
dbt-postgres>=1.10.2
dotenv>=0.9.9
psycopg2-binary>=2.9.12
sqlalchemy>=2.0.51
```

## 4. Setting Up Airflow with Docker Compose

To deploy this full developer stack, we define our environment in a `docker-compose.yaml` file. We use YAML anchors (`&airflow-common` and `*airflow-common`) to share environment variables, volumes, and networks across all Airflow services.

Add this dockerfile and docker compose file to the airflow directory [Dockerfile](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-04/files/Dockerfile) | [Docker-compose](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-04/files/docker-compose.yml)

### Mounting the dbt Project Folder
To allow Airflow to execute our dbt models, we must mount the dbt project folder from our host computer into the Airflow containers. In the `volumes` list under `x-airflow-common`inside `docker-compose.yml` file we add:
```yaml
    - Absolute/Path/To/Your/Dbt/project:/opt/airflow/dbt_warehouse
```
This maps our dbt code to `/opt/airflow/dbt_warehouse` inside all Airflow containers, giving the Airflow scheduler a fixed, reliable path to run dbt commands.

## 5. Implementing the Pipeline DAG

With Airflow 2.x and 3.x, we define our workflows using Python-based Directed Acyclic Graphs (DAGs). We use the `@dag` decorator to structure the pipeline, while using classic operators like `BashOperator` to execute individual tasks.

Our pipeline performs two steps sequentially every hour:
1. **Ingest Token Prices**: Uses the `BashOperator` to run the `ingest_prices.py` script to fetch hourly prices from DefiLlama for the execution window.

2. **Build Transformations**: Uses the `BashOperator` to navigate to run `dbt build --profiles-dir`

### Setup Instructions

1. Inside your `airflow` workspace directory, create a new directory named `dags`.

2. Add the `ingest_prices.py` script to this newly created `dags` directory.Copy the code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-04/files/ingest_prices.py)

3. In the same `dags` directory, create a new Python file (e.g., `ethereum_transfers_pipeline.py`) and paste the following DAG implementation into it: find code [here](https://github.com/Analytic-Sages/data-engineering-course/blob/main/Module-04/files/flow.py)

```python
from datetime import datetime, timedelta
from airflow.decorators import dag
from airflow.operators.bash import BashOperator

default_args = {
    'owner': 'data_engineering_team',
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

@dag(
    dag_id='ethereum_transfers_pipeline',
    default_args=default_args,
    description='An hourly pipeline to ingest token prices and build dbt transformations',
    schedule='@hourly',                      # Runs once every hour
    start_date=datetime(2023, 8, 26),        # Date of the first block processed
    catchup=False,                            # Auto-backfills historical runs
    tags=['blockchain', 'dbt'],
)
def ethereum_transfers_pipeline():

    # Task 1: Ingest Token Prices (Using BashOperator to call our ingest script)
    # We pass the dynamic hourly timestamps directly using Airflow's Jinja templates
    ingest_prices = BashOperator(
        task_id='ingest_token_prices',
        bash_command=(
            'python3 /opt/airflow/dags/ingest_prices.py '
            '--start {{ data_interval_start.int_timestamp }} '
            '--end {{ data_interval_end.int_timestamp }}'
        )
    )

    # Task 2: Run and Test dbt Models
    # Runs 'dbt build' in the mounted warehouse folder pointing to the project profile
    dbt_transform = BashOperator(
        task_id='dbt_transformations',
        bash_command='cd /opt/airflow/dbt_warehouse && dbt build --profiles-dir .'
    )

    # Setting Task Dependencies
    # Prices must be populated in the DB before dbt joins them
    ingest_prices >> dbt_transform

# Instantiate the DAG
pipeline = ethereum_transfers_pipeline()



## 6. Running Airflow and Triggering the DAG in the UI

Now that the configuration files, ingestion script, and DAG are in place, we can spin up the Airflow stack and trigger our pipeline from the Web UI.

---

### Step A: Start the Airflow Services
Open a terminal, navigate to your `airflow` directory, and run the following command to start all containers:
```bash
docker compose up
```

### Step B: Access the Airflow Web UI
1. Open your web browser and navigate to: **`http://localhost:8080`**
2. Log in using the default admin credentials:
   - **Username**: `airflow`
   - **Password**: `airflow`

### Step C: Find the DAG Using Tags
In production environments, Airflow instances can run dozens or hundreds of DAGs. To quickly locate related pipelines, we assign metadata tags to them.

1. On the Airflow home page, go to the **DAGs** tab.
2. Locate the **"Filter by tag"** search bar at the top of the DAG list.
3. Search for and select the tag **`blockchain`** or **`dbt`**.

> [NOTE]
> **Why do we use these tags?**  
> We define tags (like `tags=['blockchain', 'dbt']` in our DAG decorator) to categorize pipelines by domain or technology stack. This allows different engineering teams to easily filter, monitor, and group their specific workflows without cluttering the main dashboard.

### Step D: Activate and Trigger the DAG
1. Once filtered, click on the **`ethereum_transfers_pipeline`** DAG link to open its details view.
2. In the top-right corner of the DAG screen, click the **Play icon (Trigger)** button and select **Trigger DAG**.
3. Go to the **Grid** or **Graph** view to monitor your two tasks (`ingest_token_prices` and `dbt_transformations`) as they execute sequentially.
```

## Summary

Using Docker Compose volume mounting, we have mapped our local project codebase cleanly into the containerized environment. Airflow coordinates the ingestion script and dbt transformations sequentially, ensuring our staging, intermediate, and marts tables remain updated and tested automatically hour by hour.